# Interesting Trend
How do top amenity characteristics vary between specific neighborhoods?

In [ ]:
import pandas as pd
import json, re

berlin = pd.read_csv('https://raw.githubusercontent.com/HereOnGithub/airbnb-dashboard-2026/main/data/zip_berlin_listings.csv')
munich = pd.read_csv('https://raw.githubusercontent.com/HereOnGithub/airbnb-dashboard-2026/main/data/zip_munich_listings.csv')

In [ ]:
def parse_amenities(raw):
    if pd.isna(raw): return []
    try: return json.loads(raw.replace('""', '"'))
    except:
        m = re.findall(r'"([^"]+)"', raw)
        return m if m else []

def top_amenities_by_hood(df, hood, n=5):
    df = df[df['neighbourhood_cleansed'] == hood].copy()
    df['price'] = df['price'].replace('[\€\$,]', '', regex=True).astype(float, errors='ignore')
    df = df[pd.to_numeric(df['price'], errors='coerce') > 0]
    df['rating'] = pd.to_numeric(df['review_scores_rating'], errors='coerce')
    df['revenue'] = pd.to_numeric(df['estimated_revenue_l365d'], errors='coerce').fillna(0)
    
    stats = {}
    for _, row in df.iterrows():
        for a in parse_amenities(row.get('amenities', '')):
            a = a.strip()
            if not a: continue
            if a not in stats: stats[a] = {'rev': 0, 'rat': 0, 'count': 0, 'rat_count': 0}
            stats[a]['rev'] += row['revenue']
            stats[a]['count'] += 1
            if pd.notna(row['rating']):
                stats[a]['rat'] += row['rating']
                stats[a]['rat_count'] += 1
    
    total = len(df)
    results = []
    for name, s in stats.items():
        if s['count'] >= 5 and s['count'] < total * 0.5 and s['rat_count'] > 0 and (s['rat']/s['rat_count']) >= 4.8:
            results.append({'Amenity': name, 'Avg Rating': round(s['rat']/s['rat_count'], 2), 'Avg Revenue': round(s['rev']/s['count']), 'Listings': s['count']})
    
    result = pd.DataFrame(results).sort_values('Avg Revenue', ascending=False).head(n).reset_index(drop=True)
    result.index += 1
    print(f'--- {hood} ({total} listings) ---')
    print(result)
    print()

In [ ]:
# Berlin neighborhoods
print('=== BERLIN ===')
top_amenities_by_hood(berlin, 'Tempelhofer Vorstadt')
top_amenities_by_hood(berlin, 'südliche Luisenstadt')

In [ ]:
# Munich neighborhoods
print('=== MUNICH ===')
top_amenities_by_hood(munich, 'Ludwigsvorstadt-Isarvorstadt')
top_amenities_by_hood(munich, 'Schwabing-West')